In [8]:
!python -m pip install -U python-woc pandas matplotlib
# please clear the output of this cell in your notebook before checking it in

You should consider upgrading via the '/home/bslessma/fdac26/MiniProject2/.venv/bin/python -m pip install --upgrade pip' command.


In [9]:
from woc.remote import WocMapsRemote
from tqdm import tqdm
import pandas as pd
# creates the client
woc = WocMapsRemote( base_url="https://worldofcode.org/api/")
# if you got an API key
# woc = WocMapsRemote( base_url="https://worldofcode.org/api/", api_key="woc-XXXXXX-YYYYYY" )

# Now for each of the ten projects you were assigned get commits, e.g.
projects = [
            'intelowlproject/IntelOwl',
            'intellistream/AllianceDB',
            'yesint/pteros',
            'cvnlab/analyzePRF',
            'simonfuhrmann/mve',
            'great-expectations/great_expectations',
            'google/grr',
            'projectcypress/cypress',
            'HyperHDG/HyperHDG',
            'MIT-SPARK/TEASER-plusplus'
            ]
list_df_commits = []
for prj in projects:
  # Convert GitHub repository names to WoC V2412 project names
  prj = prj.lower().replace('/','_',2)
  commits = woc.get_values('p2c', prj)
  df = pd.DataFrame(commits, columns=["sha1"])
  df['project'] = prj
  list_df_commits.append(df)
df = pd.concat(list_df_commits)
#perhaps save the list (if no errors) so you do not need to retrieve them again
df.to_csv('df_commits.csv')
df.head(1)

,sha1,project
0,0003ffda93da9378c9d9fe8dc0b13cc05a3b71e0,intelowlproject_intelowl


In [10]:
# If the number of commits its not very large, you can try to get them all at the same time,
# The max batch size is 10
import time
# let us first split df['sha1'] in chunks
chunks = [df['sha1'][x:x+10] for x in range(0, len(df), 10)]
commit_data = []

for chunk in tqdm(chunks): # iterate over the commits
  # res, err = woc.show_content_many('commit',chunk.to_list())

  # commit.tch returns the same data but faster
  res, err = woc.get_values_many('commit.tch',chunk.to_list())
  res = {k: v[0] for k, v in res.items()}  # this conversion is necessary because of the internal implementation

  # to walk around the rate limit
  time.sleep(1)

  if err: # check for errors
    print('Got Errors', err)

  for commit_sha, commit in res.items():
    # flatten commit objects
    commit_data.append({
          'commit': commit_sha,
          'tree': commit[0],
          'parent': list(commit[1]),
          'author': commit[2][0],
          'author_time': int(commit[2][1]),
          'author_tz': commit[2][2],
          'committer': commit[3][0],
          'committer_time': int(commit[3][1]),
          'committer_tz': commit[3][2],
          'message': commit[4],
    })

df_commit_data = pd.DataFrame(commit_data)
df_commit_data = df_commit_data.merge(df, left_on='commit', right_on='sha1')
df_commit_data.to_csv('df_commit_data.csv', index=False)
df_commit_data.head(2)

  9%|▉         | 1218/12945 [20:49<3:21:29,  1.03s/it]

Got Errors {'14795024f24e3e3253e78e9a572698df7908eb71': 'Key 14795024f24e3e3253e78e9a572698df7908eb71 not found in /da5_fast/All.sha1c/commit_20.tch'}


 11%|█         | 1415/12945 [24:12<3:20:26,  1.04s/it]

Got Errors {'40112a43f6cf83cd7ce799344da4733aac174a27': 'Key 40112a43f6cf83cd7ce799344da4733aac174a27 not found in /da5_fast/All.sha1c/commit_64.tch'}


 11%|█▏        | 1464/12945 [25:03<3:16:46,  1.03s/it]

...

Got Errors {'fc7443622d5b8e4b1cbb2cc03fb8329dc46abeed': 'Key fc7443622d5b8e4b1cbb2cc03fb8329dc46abeed not found in /da5_fast/All.sha1c/commit_124.tch'}


 98%|█████████▊| 12686/12945 [3:38:24<04:26,  1.03s/it]

Got Errors {'ff2074df3a6ddf4a1259a6810b875a9436aecfc2': 'Key ff2074df3a6ddf4a1259a6810b875a9436aecfc2 not found in /da5_fast/All.sha1c/commit_127.tch'}


100%|██████████| 12945/12945 [3:42:52<00:00,  1.03s/it]


,commit,tree,parent,author,author_time,author_tz,committer,committer_time,committer_tz,message,sha1,project
0,0003ffda93da9378c9d9fe8dc0b13cc05a3b71e0,44f9608c756ec0b261c945130a440622f9752b5f,[096d5085a900a3ff03b86e0856dc35b1a595becf],a-rash <ashirrasheed96@gmail.com>,1675372411,+0400,a-rash <ashirrasheed96@gmail.com>,1675373255,+0400,fix: update code base to reflect change in tlp...,0003ffda93da9378c9d9fe8dc0b13cc05a3b71e0,intelowlproject_intelowl
1,000e0e7dea030138b58c620f37e307c5353c02ea,8baed47a846ef3906d27ae9c88ee123b6a945e83,[a5ea55eb255aa694bb4a49dcb1e14fa864ab933c],mlodic <30625432+mlodic@users.noreply.github.com>,1668081200,+0100,GitHub <noreply@github.com>,1668081200,+0100,Update api_app/analyzers_manager/observable_an...,000e0e7dea030138b58c620f37e307c5353c02ea,intelowlproject_intelowl


In [11]:
# what does the first record looks like?
# commit  parents
v = commit_data[0]
# commit tree, [parent{s}], [author, unixtime, tz ], [ committer, unixtime, tz ], Commit_message ]
print(v)

{'commit': '0003ffda93da9378c9d9fe8dc0b13cc05a3b71e0', 'tree': '44f9608c756ec0b261c945130a440622f9752b5f', 'parent': ['096d5085a900a3ff03b86e0856dc35b1a595becf'], 'author': 'a-rash <ashirrasheed96@gmail.com>', 'author_time': 1675372411, 'author_tz': '+0400', 'committer': 'a-rash <ashirrasheed96@gmail.com>', 'committer_time': 1675373255, 'committer_tz': '+0400', 'message': 'fix: update code base to reflect change in tlp standard (closes #1190)\n'}


In [12]:
# Now that the data is retrieved, save it and commit to your GH fork
# First, we need to flatten info in order to export as csv
# our dataframe will have columns project,'commit, author, time, message'
dfinf = pd.DataFrame(columns=['project', 'commit', 'author', 'time', 'message'])
for k in commit_data:
  row = pd.Series({'project':prj, 'commit': k['commit'], 'author': k['author'], 'time': k['author_time'], 'message':k['message']})
  dfinf = pd.concat([dfinf, row.to_frame().T ], ignore_index=True)


In [13]:
# check if it has the right content
dfinf.head(1)

,project,commit,author,time,message
0,mit-spark_teaser-plusplus,0003ffda93da9378c9d9fe8dc0b13cc05a3b71e0,a-rash <ashirrasheed96@gmail.com>,1675372411,fix: update code base to reflect change in tlp...


In [14]:
#mode a means append, so you have all your projects in the same file
yournetid='bslessma'
dfinf.to_csv(yournetid+'_project_summary.csv', index=False,sep=';', mode='a', header=False)

# Make sure you check in to your fork not just the notebook but also the csv files!!!

# Don't forget to add requested data from github and this notebook
### For each of the 10 projects go to their github repo and get the number of stars, number of forks, and the last commit date
### Report (in your notebook) the number of commits, the number of authors, and max and min time for each project based on WoC commits and also add the info you obtained from github